In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv(r"D:\bank\loan.csv", low_memory=False)


In [2]:
df.head(2)

,loan_amnt,int_rate,grade,emp_length,annual_inc,loan_status,purpose,addr_state,issue_d,term,default_flag,income_group,loan_size,risk_tier,issue_year
0,3600.0,13.99,C,10+ years,55000.0,Fully Paid,debt_consolidation,PA,Dec-2015,36 months,0,Medium (40-80K),Small (<5K),Medium Risk,2015
1,24700.0,11.99,C,10+ years,65000.0,Fully Paid,small_business,SD,Dec-2015,36 months,0,Medium (40-80K),Large (15-25K),Medium Risk,2015


In [3]:
print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")

Rows: 1,345,310  |  Columns: 15


cleaning the Data

In [4]:
print(df.isnull().sum().sort_values(ascending=False).head(10))

emp_length      78511
income_group      361
int_rate            0
grade               0
annual_inc          0
loan_status         0
loan_amnt           0
purpose             0
addr_state          0
term                0
dtype: int64


In [5]:
df["loan_status"].value_counts()

loan_status
Fully Paid     1076751
Charged Off     268559
Name: count, dtype: int64

In [6]:
df["grade"].value_counts().sort_index()

grade
A    235090
B    392741
C    381686
D    200953
E     93650
F     32058
G      9132
Name: count, dtype: int64

In [7]:
df["loan_amnt"].describe().round(2)

count    1345310.00
mean       14419.97
std         8717.05
min          500.00
25%         8000.00
50%        12000.00
75%        20000.00
max        40000.00
Name: loan_amnt, dtype: float64

In [8]:
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])]

In [9]:
cols = ["loan_amnt", "int_rate", "grade", "emp_length",
        "annual_inc", "loan_status", "purpose",
        "addr_state", "issue_d", "term"]
cols = [c for c in cols if c in df.columns]
df = df[cols].copy()

In [10]:
if df["int_rate"].dtype == object:
    df["int_rate"] = df["int_rate"].str.replace("%", "").astype(float)

In [11]:
df["loan_amnt"]  = pd.to_numeric(df["loan_amnt"],  errors="coerce")
df["annual_inc"] = pd.to_numeric(df["annual_inc"], errors="coerce")

In [12]:
df.dropna(subset=["loan_amnt", "annual_inc",
                  "grade", "loan_status"], inplace=True)

print(f"Clean rows : {len(df):,}")
print(f"Null values: {df.isnull().sum().sum()}")

Clean rows : 1,345,310
Null values: 78511


Add Calculated Columns

In [13]:
# Default flag — 1 = defaulted, 0 = paid
df["default_flag"] = (
    df["loan_status"] == "Charged Off"
).astype(int)

In [14]:
# Income group segmentation
df["income_group"] = pd.cut(
    df["annual_inc"],
    bins=[0, 40000, 80000, 150000, float("inf")],
    labels=["Low (<40K)", "Medium (40-80K)",
            "High (80-150K)", "Very High (150K+)"]
)

In [15]:
# Loan size category
df["loan_size"] = pd.cut(
    df["loan_amnt"],
    bins=[0, 5000, 15000, 25000, float("inf")],
    labels=["Small (<5K)", "Medium (5-15K)",
            "Large (15-25K)", "Very Large (25K+)"]
)

In [16]:
# Risk tier based on grade
df["risk_tier"] = df["grade"].map({
    "A": "Low Risk", "B": "Low Risk",
    "C": "Medium Risk", "D": "Medium Risk",
    "E": "High Risk", "F": "High Risk",
    "G": "High Risk"
})


In [17]:
# Issue year
try:
    df["issue_year"] = pd.to_datetime(
        df["issue_d"], format="%b-%Y"
    ).dt.year
except:
    df["issue_year"] = 2016

print("Added: default_flag, income_group, loan_size,")
print("       risk_tier, issue_year")

Added: default_flag, income_group, loan_size,
       risk_tier, issue_year



Key Analysis

In [18]:
# Overall KPIs
total_loans     = len(df)
total_defaults  = df["default_flag"].sum()
default_rate    = df["default_flag"].mean() * 100
avg_loan        = df["loan_amnt"].mean()
avg_int_rate    = df["int_rate"].mean()

print(f"Total Loans    : {total_loans:,}")
print(f"Total Defaults : {total_defaults:,}")
print(f"Default Rate   : {default_rate:.2f}%")
print(f"Avg Loan Amount: ${avg_loan:,.0f}")
print(f"Avg Int Rate   : {avg_int_rate:.2f}%")

Total Loans    : 1,345,310
Total Defaults : 268,559
Default Rate   : 19.96%
Avg Loan Amount: $14,420
Avg Int Rate   : 13.24%


In [19]:
# Default rate by grade
grade_summary = df.groupby("grade").agg(
    Total_Loans   = ("loan_amnt",    "count"),
    Defaults      = ("default_flag", "sum"),
    Avg_Loan      = ("loan_amnt",    "mean"),
    Avg_Int_Rate  = ("int_rate",     "mean")
).round(2)
grade_summary["Default_Rate_%"] = (
    grade_summary["Defaults"] /
    grade_summary["Total_Loans"] * 100
).round(2)
print(grade_summary.to_string())


       Total_Loans  Defaults  Avg_Loan  Avg_Int_Rate  Default_Rate_%
grade                                                               
A           235090     14201  13892.35          7.11            6.04
B           392741     52569  13237.63         10.68           13.39
C           381686     85649  14188.43         14.02           22.44
D           200953     61054  15272.41         17.72           30.38
E            93650     36035  17618.05         21.14           38.48
F            32058     14491  19087.56         24.93           45.20
G             9132      4560  20588.81         27.73           49.93


In [20]:
df.to_csv(r"D:\bank\loan.csv", index=False)
print("Saved! Load this into Power BI.")

Saved! Load this into Power BI.


In [21]:
grade_summary.to_csv(r"D:\bank\default_by_grade.csv")

print(f"Rows   : {len(df):,}")
print(f"Columns: {list(df.columns)}")

Rows   : 1,345,310
Columns: ['loan_amnt', 'int_rate', 'grade', 'emp_length', 'annual_inc', 'loan_status', 'purpose', 'addr_state', 'issue_d', 'term', 'default_flag', 'income_group', 'loan_size', 'risk_tier', 'issue_year']
